In [34]:
import pandas as pd

df = pd.read_csv("retail-orders-raw.csv")

print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

df.head()

Rows: 12
Columns: 9


,order_id,order_date,customer_segment,city,category,quantity,unit_price,discount_pct,payment_status
0,RT-1001,2026-01-03,Student,Chennai,Learning Kit,2,799,10.0,Paid
1,RT-1002,03/01/2026,Fresher,Bengaluru,Course Access,1,1499,0.0,paid
2,RT-1003,2026-01-05,student,Chennai,Course Access,1,1499,NaN,Pending
3,RT-1004,2026-01-07,Professional,Hyderabad,Learning Kit,3,799,5.0,Paid
4,RT-1004,2026-01-07,Professional,Hyderabad,Learning Kit,3,799,5.0,Paid


In [35]:
# Completeness check: missing values in each column

missing_values = df.isnull().sum()

print("Missing values by column:")
print(missing_values)

Missing values by column:
order_id            0
order_date          1
customer_segment    0
city                1
category            0
quantity            0
unit_price          0
discount_pct        1
payment_status      0
dtype: int64


In [36]:
# Uniqueness check for order_id

duplicate_orders = df["order_id"].duplicated().sum()

print("Duplicate order IDs:", duplicate_orders)

print("\nDuplicate order records:")
print(df[df["order_id"].duplicated(keep=False)])

Duplicate order IDs: 1

Duplicate order records:
  order_id  order_date customer_segment       city      category quantity  \
3  RT-1004  2026-01-07     Professional  Hyderabad  Learning Kit        3   
4  RT-1004  2026-01-07     Professional  Hyderabad  Learning Kit        3   

   unit_price  discount_pct payment_status  
3         799           5.0           Paid  
4         799           5.0           Paid  


In [37]:
# Validity checks

# Convert quantity to numeric for validation
quantity_numeric = pd.to_numeric(df["quantity"], errors="coerce")

print("Invalid quantity values:")
print(df[quantity_numeric <= 0][["order_id", "quantity"]])

print("\nNon-numeric quantity values:")
print(df[quantity_numeric.isna()][["order_id", "quantity"]])

print("\nInvalid discount values:")
print(df[(df["discount_pct"] < 0) | (df["discount_pct"] > 100)][["order_id", "discount_pct"]])

Invalid quantity values:
  order_id quantity
6  RT-1006       -1

Non-numeric quantity values:
  order_id quantity
8  RT-1008      two

Invalid discount values:
  order_id  discount_pct
7  RT-1007         105.0


In [38]:
# Consistency checks

print("Customer segment values:")
print(df["customer_segment"].value_counts(dropna=False))

print("\nPayment status values:")
print(df["payment_status"].value_counts(dropna=False))

print("\nCategory values:")
print(df["category"].value_counts(dropna=False))

Customer segment values:
customer_segment
Student         4
Professional    4
Fresher         3
student         1
Name: count, dtype: int64

Payment status values:
payment_status
Paid        7
Pending     2
paid        1
Failed      1
Refunded    1
Name: count, dtype: int64

Category values:
category
Learning Kit      5
Course Access     4
Mentor Session    3
Name: count, dtype: int64


In [39]:
# Corrected freshness check for mixed date formats

df["order_date_parsed"] = pd.to_datetime(
    df["order_date"],
    errors="coerce",
    format="mixed"
)

print("Earliest valid order date:", df["order_date_parsed"].min())
print("Latest valid order date:", df["order_date_parsed"].max())

print("\nInvalid or missing dates:")
print(
    df[df["order_date_parsed"].isna()][
        ["order_id", "order_date"]
    ]
)

Earliest valid order date: 2026-01-03 00:00:00
Latest valid order date: 2026-03-01 00:00:00

Invalid or missing dates:
   order_id  order_date
6   RT-1006  2026-13-10
11  RT-1011         NaN


## Data Quality Summary

### Completeness
Missing values were found in order_date, city, and discount_pct.

### Uniqueness
One duplicate order ID was identified: RT-1004.

### Validity
Invalid quantity values were identified for RT-1006 (-1) and RT-1008 (non-numeric value "two"). An invalid discount value of 105% was identified for RT-1007.

### Consistency
Inconsistent capitalization was found in customer_segment (Student/student) and payment_status (Paid/paid).

### Freshness
The valid order-date range is 2026-01-03 to 2026-03-01. RT-1006 contains an invalid date (2026-13-10), and RT-1011 has a missing order date.

### Overall Assessment
The dataset requires data-quality remediation before being used for reliable KPI reporting and business analysis.

In [40]:
quality_contract = """
# Data Quality Contract

## Dataset
Retail Orders Dataset

## Decision Owner
Head of Sales / Sales Operations Manager

## 1. Completeness
Rule: Required fields should not contain missing values.
Threshold: Maximum 1% missing values.
Failure Action: Flag the dataset for review and notify the data owner.

## 2. Uniqueness
Rule: order_id must be unique.
Threshold: 0 duplicate order IDs.
Failure Action: Duplicate records must be investigated and removed or corrected before KPI reporting.

## 3. Validity
Rule: quantity must be greater than 0.
Threshold: 100% valid quantity values.
Failure Action: Invalid records must be quarantined and corrected.

Rule: discount_pct must be between 0 and 100.
Threshold: 100% valid discount values.
Failure Action: Invalid discount records must be reviewed before reporting.

## 4. Consistency
Rule: Categorical values must use consistent capitalization and approved values.
Threshold: 100% consistency.
Failure Action: Standardize inconsistent values before KPI calculation.

## 5. Freshness
Rule: order_date must be a valid date.
Threshold: 100% valid and non-missing dates.
Failure Action: Records with invalid or missing dates must be investigated before use in reporting.

## Escalation
Any quality rule failure should be reported to the Data Owner / Sales Operations Manager.
Critical failures should block the affected data from KPI reporting until remediation is completed.
"""

with open("Data_Quality_Contract.md", "w") as f:
    f.write(quality_contract)

print("Data Quality Contract created successfully.")

Data Quality Contract created successfully.
